# NLP Pipeline — Code de la Route Marocain (Loi 52.05)
### Devoir 1 · Semaine 2

**Objectif :** transformer le PDF brut (126 pages, arabe) en `export_final.csv` structuré.

**Pipeline :**
- **A** — Prétraitement Arabe : Tashkeel · Hamzas · Bidi PDF
- **B** — Extraction Règles : Regex Unicode (amende, points, prison, véhicule, tags)
- **C** — ML : TF-IDF n-grammes de caractères + K-Means clustering


In [11]:
import re, csv
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

PDF_PATH = 'code de la route MA52_05.pdf'
OUTPUT   = 'export_final.csv'
print('Librairies chargées.')

Librairies chargées.


---
## Étape A — Prétraitement Arabe
> Normalisation Unicode maison (sans PyArabic). Critique pour un moteur de recherche non biaisé.

| Problème | Solution | Plage Unicode |
|---|---|---|
| **Tashkeel** (voyelles) | Suppression | U+064B – U+065F, U+0670 |
| **Hamzas** (أ إ آ ؤ ئ) | → forme canonique | U+0623, U+0625, U+0622... |
| **Ta Marbuta** (ة) | → ه (optionnel NER) | U+0629 |
| **Bidi bidi PDF InDesign** | Suppression LRE/PDF | U+202A – U+202E |

In [12]:
# Normalisation Hamzas : أ إ آ → ا  |  ؤ → و  |  ئ → ي
HAMZA_TABLE = str.maketrans({
    'أ': 'ا', 'إ': 'ا', 'آ': 'ا',
    'ؤ': 'و', 'ئ': 'ي',
})

# Tashkeel : U+064B→U+065F (Fatha/Damma/Kasra/Shadda/Sukun)
TASHKEEL_RE = re.compile(r'[\u064B-\u065F\u0670\u0610-\u061A]')

# Bidi control chars (artefacts PDF InDesign RTL)
BIDI_RE = re.compile(r'[\u200F\u200E\u202A-\u202E\u200B-\u200D\uFEFF\u200C]')


def normalize_arabic(text: str, keep_ta_marbuta: bool = False) -> str:
    """
    1. Suppression bidi (artefacts PDF InDesign)
    2. Suppression Tashkeel → évite le biais du moteur de recherche
       ex: 'سَياقة' et 'سياقة' doivent matcher
    3. Normalisation Hamzas → forme canonique
    4. Ta Marbuta optionnel : ة → ه (pour NER/matching)
    5. Collapse espaces multiples
    """
    if not text:
        return ""
    text = BIDI_RE.sub('', text)           # 1. bidi
    text = TASHKEEL_RE.sub('', text)       # 2. voyelles
    text = text.translate(HAMZA_TABLE)     # 3. Hamzas
    if not keep_ta_marbuta:
        text = text.translate({0x629: 0x647})  # ة → ه
    return re.sub(r'[ \t]+', ' ', text).strip()


# Test
avant = 'رُخْصَةُ السِّيَاقَةِ'
print(f'Avant : {avant}')
print(f'Après : {normalize_arabic(avant)}')

Avant : رُخْصَةُ السِّيَاقَةِ
Après : رخصه السياقه


---
## Étape B — Extraction par Patterns (Rules-based NLP)
> Le langage juridique est très codifié → Regex Unicode prioritaire.
> Patterns construits sur l'analyse directe du PDF.

In [13]:
# ── B1 : Amende en dirhams ───────────────────────────────────────────────
# Stratégie : chiffres >= 200 avant 'درهم'
# Seuil 200 DH filtre les numéros de page (1-126) et d'article
PATTERN_AMENDE = re.compile(
    r'\(?([\d.,]{3,})\)?'
    r'\s*(?:إلى\s*\(?([\d.,]{3,})\)?)?'
    r'\s*(?:درهم|dirhams)',
    re.UNICODE
)

# Montants en toutes lettres + 'درهم' dans rayon 80 chars
AMENDE_LITTERALE = {
    'ألف ومائتي': 1200,  'اثني عشر ألف': 12000,
    'خمسة وثلاثين ألف': 35000, 'مائة ألف': 100000,
    'عشرين ألف': 20000,  'ثمانية آلاف': 8000,
    'خمسة آلاف': 5000,   'أربعة آلاف': 4000,
    'ثلاثة آلاف': 3000,  'ألفين': 2000, 'ألفي': 2000,
    'خمسمائة': 500,      'ألف': 1000,
}

def _to_int(s):
    c = re.sub(r'[.,\s]', '', s)
    return int(c) if c and c.isdigit() else None

def extract_amende(text):
    """Extrait (amende_min, amende_max) en DH. Deux passes."""
    amounts = []
    for m in PATTERN_AMENDE.finditer(text):
        for g in m.groups():
            if g:
                v = _to_int(g)
                if v and v >= 200: amounts.append(v)
    for phrase, valeur in sorted(AMENDE_LITTERALE.items(), key=lambda x: -len(x[0])):
        pat = re.compile(re.escape(phrase) + r'.{0,80}درهم', re.UNICODE|re.DOTALL)
        if pat.search(text): amounts.append(valeur)
    if not amounts: return None, None
    amounts = sorted(set(amounts))
    return amounts[0], amounts[-1] if len(amounts) > 1 else None

print('Pattern amende OK')

Pattern amende OK


In [14]:
# ── B2 : Points de retrait : خصم N نقط ──────────────────────────────────
PAT_POINTS = re.compile(
    r'(?:خصم|يخصم|فقدان|سحب)\s*(?:\S+\s+){0,4}?(?P<pts>\d+)\s*(?:نقط|نقطة|نقاط)',
    re.UNICODE
)
PAT_POINTS_PAREN = re.compile(r'\((?P<pts>\d+)\)\s*(?:نقط|نقطة)', re.UNICODE)

def extract_points(text):
    m = PAT_POINTS.search(text) or PAT_POINTS_PAREN.search(text)
    return int(m.group('pts')) if m else None


# ── B3 : Prison : الحبس من X إلى Y ──────────────────────────────────────
PAT_PRISON = re.compile(
    r'(?:الحبس|بالحبس)\s+من\s+(?P<min>[^إ]+?)\s+إلى\s+(?P<max>[^\n،.]{3,40})',
    re.UNICODE
)
def extract_prison(text):
    m = PAT_PRISON.search(text)
    return f"من {m.group('min').strip()} إلى {m.group('max').strip()}" if m else None


# ── B4 : Catégorie véhicule ───────────────────────────────────────────────
VEHICLE_PATTERNS = [
    ('poids_lourd', r'(?:شاحنة|سيارة نقل|مركبة ثقيل|نقل البضاعة)'),
    ('moto',        r'(?:دراجة نارية|دراجة آلية|موتور)'),
    ('taxi',        r'(?:سيارة أجرة|تاكسي)'),
    ('bus',         r'(?:حافلة|باص|عربة نقل الأشخاص)'),
    ('agricole',    r'(?:مركبة فلاحية|جرار|غابوية)'),
]
def extract_vehicle_category(text):
    for cat, pat in VEHICLE_PATTERNS:
        if re.search(pat, text, re.UNICODE): return cat
    return 'tous_vehicules' if re.search(r'مركبة|سيارة', text) else 'non_spécifié'


# ── B5 : Tags thématiques ─────────────────────────────────────────────────
KEYWORD_DICT = {
    'vitesse':       r'سرعة|تجاوز السرعة|مقياس السرعة',
    'alcool':        r'كحول|سكر|تحت تأثير',
    'ceinture':      r'حزام الأمان|حزام السلامة',
    'telephone':     r'هاتف|جوال|اتصال',
    'stationnement': r'توقف|وقوف|انتظار|ركن المركبة',
    'priorité':      r'أولوية المرور|تقديم الأولوية',
    'feu_rouge':     r'ضوء أحمر|إشارة مرور|إشارة ضوئية',
    'dépassement':   r'تجاوز|سبق السيارة',
    'nuit':          r'ليل|ليلا|مصابيح|أضواء',
    'autoroute':     r'طريق سريع|أوتوستراد',
    'piéton':        r'راجل|مشاة|عابر طريق',
    'fuite':         r'فرار|هرب|عدم التوقف عند الحادث',
    'assurance':     r'تأمين|بوليصة',
    'permis':        r'رخصة السياقة|إجازة القيادة',
    'accident':      r'حادث|اصطدام|إصابة',
    'signalisation': r'إشارة|علامة|لوحة',
}
def extract_keywords(text):
    return [kw for kw, pat in KEYWORD_DICT.items() if re.search(pat, text, re.UNICODE)]


# ── B6 : Drapeaux booléens ────────────────────────────────────────────────
PAT_RECIDIVE   = re.compile(r'حالة العود|في حال العود', re.UNICODE)
PAT_SUSPENSION = re.compile(r'(?:توقيف|سحب|إلغاء)\s+رخصة السياقة', re.UNICODE)
PAT_IMMOB      = re.compile(r'(?:حجز|إيداع)\s+(?:المركبة|السيارة)', re.UNICODE)
PAT_INTERDIT   = re.compile(r'(?:الحرمان|المنع)\s+من\s+الحصول.{0,30}رخصة', re.UNICODE)

print('Tous les patterns Regex définis.')

Tous les patterns Regex définis.


---
## Étape C — TF-IDF + K-Means (Approche ML)
> N-grammes de caractères (2–4) : robustes pour la morphologie arabe **sans tokenizer spécialisé**.
> Pourquoi `char_wb` ?
> - L'arabe a des racines trilittères avec des patterns morphologiques riches
> - Les n-grammes capturent ces radicaux directement
> - Pas besoin de stemmer ou d'analyseur morphologique externe

In [15]:
# ── C1 : Classification par rôle (Approche 1 — règles) ──────────────────
ROLE_SEEDS = {
    'définition':   ['يقصد', 'يعني', 'لأغراض هذا القانون'],
    'obligation':   ['يجب', 'يتعين', 'يلتزم'],
    'interdiction': ['يحظر', 'يمنع', 'لا يجوز'],
    'sanction':     ['يعاقب', 'غرامة', 'حبس', 'درهم'],
    'procédure':    ['يقدم', 'يودع', 'طلب', 'مسطرة'],
}
def classify_role(text):
    scores = {r: sum(1 for s in seeds if s in text)
              for r, seeds in ROLE_SEEDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'autre'


# ── C2 : TF-IDF + K-Means (Approche 2 — ML) ──────────────────────────────
class TFIDFKMeans:
    """
    TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4)) + KMeans.

    Avantages pour l'arabe :
      - char_wb capture les radicaux trilittères sans tokenizer
      - sublinear_tf atténue les termes très fréquents (المادة, يعاقب)
      - min_df=2 supprime le bruit (hapax)
    """
    THEME_SEEDS = [
        ('رخصة', 'permis_conduite'), ('سرعة', 'vitesse_excès'),
        ('وقوف', 'stationnement'),   ('تأمين', 'assurance_technique'),
        ('كحول', 'alcool_stupéfiants'), ('حادث', 'accidents'),
        ('نقل',  'transport_marchandises'), ('جنحة', 'infractions_pénales'),
    ]
    def __init__(self, n_clusters=8):
        self.n_clusters = n_clusters
        self.vec = TfidfVectorizer(
            analyzer='char_wb', ngram_range=(2, 4),
            max_features=3000, sublinear_tf=True, min_df=2
        )
        self.km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        self.labels = {}; self.fitted = False

    def fit(self, texts):
        X = normalize(self.vec.fit_transform(texts))
        self.km.fit(X)
        features = self.vec.get_feature_names_out()
        for cid in range(self.n_clusters):
            top = [features[i] for i in
                   self.km.cluster_centers_[cid].argsort()[-8:][::-1]]
            label = next(
                (th for seed, th in self.THEME_SEEDS
                 if any(seed[:4] in f for f in top)),
                f'cluster_{cid}'
            )
            self.labels[cid] = label
        self.fitted = True

    def predict(self, text):
        if not self.fitted: return 'non_entraîné', -1
        cid = int(self.km.predict(normalize(self.vec.transform([text])))[0])
        return self.labels.get(cid, f'cluster_{cid}'), cid

print('Classifieurs définis.')

Classifieurs définis.


---
## Extraction PDF + Segmentation + Entraînement

In [16]:
# ── Extraction texte PDF ────────────────────────────────────────────────
reader = PdfReader(PDF_PATH)
raw = '\n'.join(page.extract_text() or '' for page in reader.pages)
print(f'Texte extrait : {len(raw):,} caractères ({len(reader.pages)} pages)')

# ── Segmentation en articles ─────────────────────────────────────────────
# Nettoyer les bidi AVANT le split (le PDF InDesign encode les chiffres
# latins entre marqueurs LRE/PDF qui peuvent perturber le regex)
text_clean = BIDI_RE.sub('', raw)

split_re = re.compile(r'(ا[لم]{1,2}ادة?\s*\d+(?:-\d+)?)', re.UNICODE)
parts    = split_re.split(text_clean)

articles = []
for i in range(1, len(parts), 2):
    num_m = re.search(r'(\d+(?:-\d+)?)', parts[i])
    if num_m:
        articles.append({'id': num_m.group(1),
                         'body': parts[i+1].strip() if i+1 < len(parts) else ''})

print(f'Articles détectés : {len(articles)}')

# ── Entraînement TF-IDF K-Means ──────────────────────────────────────────
corpus  = [normalize_arabic(a['body'], keep_ta_marbuta=True)
           for a in articles if len(a['body']) > 30]
n_clust = min(10, max(2, len(corpus) // 4))
clf     = TFIDFKMeans(n_clusters=n_clust)
clf.fit(corpus)
print(f'K-Means : k={n_clust}, labels={list(clf.labels.values())}')

Texte extrait : 216,951 caractères (126 pages)
Articles détectés : 79
K-Means : k=10, labels=['cluster_0', 'cluster_1', 'cluster_2', 'cluster_3', 'cluster_4', 'cluster_5', 'cluster_6', 'cluster_7', 'cluster_8', 'cluster_9']


In [17]:
def process_article(art, clf):
    raw  = art['body']
    norm = normalize_arabic(raw, keep_ta_marbuta=True)

    # Description : premier énoncé pénal (يعاقب / يحظر / يمنع)
    desc_m = re.search(
        r'(?:يعاقب|يحظر|يمنع|لا يجوز|يجب)\s+.{10,400}?(?=\n|\.|؛)',
        norm, re.UNICODE|re.DOTALL
    )
    desc = desc_m.group(0).strip() if desc_m else norm[:300]

    amende_min, amende_max     = extract_amende(raw)
    cluster_label, cluster_id  = clf.predict(norm)

    return {
        'article_id':              art['id'],
        'infraction_desc':         desc[:500],
        'categorie_vehicule':      extract_vehicle_category(raw),
        'amende_min_dh':           amende_min or '',
        'amende_max_dh':           amende_max or '',
        'points_retrait':          extract_points(raw) or '',
        'peine_prison':            extract_prison(raw) or '',
        'recidive_prevue':         'oui' if PAT_RECIDIVE.search(raw)   else 'non',
        'suspension_permis':       'oui' if PAT_SUSPENSION.search(norm) else 'non',
        'immobilisation_vehicule': 'oui' if PAT_IMMOB.search(norm)      else 'non',
        'interdiction_conduire':   'oui' if PAT_INTERDIT.search(raw)    else 'non',
        'mots_cles':               '|'.join(extract_keywords(norm)),
        'role_paragraphe_regles':  classify_role(norm),   # Approche 1
        'cluster_thematique_ml':   cluster_label,          # Approche 2
        'cluster_id':              cluster_id,
        'texte_apercu':            raw[:200].replace('\n', ' ').strip(),
    }

rows  = [process_article(a, clf) for a in articles]
stats = {
    'amende':     sum(1 for r in rows if r['amende_min_dh']),
    'points':     sum(1 for r in rows if r['points_retrait']),
    'prison':     sum(1 for r in rows if r['peine_prison']),
    'suspension': sum(1 for r in rows if r['suspension_permis'] == 'oui'),
    'recidive':   sum(1 for r in rows if r['recidive_prevue']   == 'oui'),
}
print(f'Articles traités : {len(rows)}')
for k, v in stats.items():
    print(f'  Avec {k:12}: {v}')

Articles traités : 79
  Avec amende      : 14
  Avec points      : 0
  Avec prison      : 9
  Avec suspension  : 14
  Avec recidive    : 22


In [18]:
# Export CSV final
# utf-8-sig = BOM UTF-8 : compatibilité Excel avec texte arabe RTL
fieldnames = [
    'article_id', 'infraction_desc', 'categorie_vehicule',
    'amende_min_dh', 'amende_max_dh', 'points_retrait', 'peine_prison',
    'recidive_prevue', 'suspension_permis', 'immobilisation_vehicule',
    'interdiction_conduire', 'mots_cles',
    'role_paragraphe_regles', 'cluster_thematique_ml',
    'cluster_id', 'texte_apercu'
]
with open(OUTPUT, 'w', newline='', encoding='utf-8-sig') as f:
    w = csv.DictWriter(f, fieldnames=fieldnames, quoting=csv.QUOTE_ALL)
    w.writeheader()
    w.writerows(rows)

print(f'Exporté : {OUTPUT}')
print(f'Lignes : {len(rows)} | Colonnes : {len(fieldnames)}')

Exporté : export_final.csv
Lignes : 79 | Colonnes : 16


---
## Aperçu et analyse des résultats

In [19]:
import pandas as pd

df = pd.read_csv(OUTPUT, encoding='utf-8-sig')

# Articles avec amendes
amendes = df[df['amende_min_dh'].notna() & (df['amende_min_dh'] != '')]
print('=== Articles avec amendes détectées ===')
display(amendes[['article_id','amende_min_dh','amende_max_dh',
                  'peine_prison','suspension_permis','recidive_prevue',
                  'mots_cles','role_paragraphe_regles']].head(8))

=== Articles avec amendes détectées ===


,article_id,amende_min_dh,amende_max_dh,peine_prison,suspension_permis,recidive_prevue,mots_cles,role_paragraphe_regles
21,49,1000.0,5000.0,من ثلاثة أشهر إلى ثلاث سنوات وبغرامة من ألفين,oui,oui,permis,sanction
22,130,1000.0,12000.0,من شهر إلى ثلاثة أشهر وبضعف الغرامة المقررة في...,oui,oui,alcool|stationnement|permis|accident|signalisa...,sanction
23,153,1000.0,100000.0,من ستة\n) درهم :5.000(\n -كل مالك مركبة خاضعة ...,non,oui,stationnement|permis,sanction
24,164,1000.0,5000.0,من شهر واحد إلى ستة,non,oui,vitesse|alcool|stationnement|dépassement|nuit|...,sanction
26,7,1000.0,NaN,NaN,oui,non,vitesse|alcool|stationnement|dépassement|nuit|...,sanction
28,7,1000.0,2000.0,من شهر واحد إلى سنتين وبغرامة من ألفين وأربعمائة,oui,non,vitesse|alcool|stationnement|dépassement|nuit|...,sanction
30,2,500.0,20000.0,من شهر إلى ثلاثة أشهر وبغرامة من ألف وخمسمائة,oui,oui,vitesse|alcool|ceinture|stationnement|dépassem...,sanction
35,3,500.0,NaN,NaN,oui,oui,stationnement|nuit|permis|accident,sanction


In [20]:
print('=== Distribution des rôles (règles) ===')
print(df['role_paragraphe_regles'].value_counts())
print()
print('=== Distribution clusters TF-IDF (ML) ===')
print(df['cluster_thematique_ml'].value_counts())

=== Distribution des rôles (règles) ===
role_paragraphe_regles
sanction        27
obligation      22
autre           16
procédure        9
interdiction     3
définition       2
Name: count, dtype: int64

=== Distribution clusters TF-IDF (ML) ===
cluster_thematique_ml
cluster_5    17
cluster_2    17
cluster_4    13
cluster_1     9
cluster_8     7
cluster_9     5
cluster_6     4
cluster_0     3
cluster_3     3
cluster_7     1
Name: count, dtype: int64


---
## Conclusion

| Critère | Approche 1 — Règles (Regex) | Approche 2 — ML (TF-IDF) |
|---|---|---|
| **Vitesse** | Très rapide, déterministe | Nécessite corpus d'entraînement |
| **Précision** | Haute sur patterns codifiés | Variable selon k et corpus |
| **Maintenabilité** | Patterns à écrire manuellement | Auto-adaptatif |
| **Cas couverts** | Amende, prison, points, véhicule | Groupes thématiques latents |
| **Recommandé pour** | Production (juridique codifié) | Exploration / découverte |



In [22]:
!pip install transformers


  Using cached transformers-5.5.4-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.11.0-py3-none-any.whl.metadata (14 kB)
  Using cached regex-2026.4.4-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached hf_xet-1.4.3-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
Using cached transformers-5.5.4-py3-none-any.whl (10.2 MB)
Using cached huggingface_hub-1.11.0-py3-none-any.whl (645 kB)
Using cached hf_xet-1.4.3-cp37-abi3-win_amd64.whl (3.7 MB)
Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)
Using cached regex-2026.4.4-cp313-cp313-win_amd64.whl (277 kB)
Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl (341 kB)

   ---------------------------------------- 0/6 [safetensors]
  Attempting uninstall: regex
   ---------------------------------------- 0/6 [safetensors]
    Found existing installation: regex 20

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
instapy 0.6.16 requires chardet<4,>=3.0.4, but you have chardet 4.0.0 which is incompatible.
instapy 0.6.16 requires jsonschema<3,>=2.6.0, but you have jsonschema 4.25.1 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("asafaya/bert-base-arabic")
model = AutoModel.from_pretrained("asafaya/bert-base-arabic")

c:\Users\pc\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pc\.cache\huggingface\hub\models--asafaya--bert-base-arabic. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4906.25it/s]
BertModel LOAD REPORT from